In [0]:
from pyspark.sql.functions import *
b = spark.table("workspace.bronze.raw_customers")

valid_cond = col("cust_id").isNotNull() & col("email").rlike(r"^[^@]+@[^@]+\.[^@]+$")
good = (b.filter(valid_cond)
         .withColumn("name", initcap(trim(col("name"))))
         .dropDuplicates(["cust_id"]))
bad = b.filter(~valid_cond).withColumn("dq_reason", lit("null_key_or_bad_email"))

bad.write.mode("append").saveAsTable("workspace.dq.quarantine_customers")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.dq")   # in case Step 1 missed it

good.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.stg_customers")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.silver.clean_customers (
  cust_id INT, name STRING, email STRING, city STRING, row_hash STRING,
  eff_start TIMESTAMP, eff_end TIMESTAMP, is_current BOOLEAN);

In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW stg_hashed AS
SELECT *, sha2(concat_ws('|', city, email), 256) AS row_hash FROM workspace.silver.stg_customers
""")

spark.sql("""
MERGE INTO workspace.silver.clean_customers t
USING (
  -- rows that close the old version (match key) ...
  SELECT s.cust_id AS merge_key, s.* FROM stg_hashed s
  UNION ALL
  -- ... plus rows that insert the new version (null key never matches)
  SELECT NULL AS merge_key, s.*
  FROM stg_hashed s
  JOIN workspace.silver.clean_customers t
    ON s.cust_id = t.cust_id AND t.is_current = true AND s.row_hash <> t.row_hash
) src
ON t.cust_id = src.merge_key AND t.is_current = true
WHEN MATCHED AND t.row_hash <> src.row_hash THEN
  UPDATE SET t.is_current = false, t.eff_end = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (cust_id, name, email, city, row_hash, eff_start, eff_end, is_current)
  VALUES (src.cust_id, src.name, src.email, src.city, src.row_hash,
          current_timestamp(), TIMESTAMP'9999-12-31', true)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
SELECT * FROM workspace.silver.clean_customers WHERE cust_id = 101 ORDER BY eff_start;

cust_id,name,email,city,row_hash,eff_start,eff_end,is_current
101,Michael Elliott,dennisacosta@example.com,Davisbury,e8053dba51837a6c6a0b167f32c04a88dba2e1ea0623c505810c848bf9eaf3bc,2026-11-07T14:07:04.433Z,9999-12-31T00:00:00.000Z,true


In [0]:
from delta.tables import DeltaTable
src = (spark.table("workspace.bronze.raw_orders")
       .filter("last_updated_ts >= current_date() - INTERVAL 3 DAYS")   # lookback
       .dropDuplicates(["order_id"]))
tgt = DeltaTable.forName(spark, "workspace.silver.clean_orders") if spark.catalog.tableExists("workspace.silver.clean_orders") else None
if tgt is None:
    src.write.saveAsTable("workspace.silver.clean_orders")
else:
    (tgt.alias("t").merge(src.alias("s"), "t.order_id = s.order_id")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# (bronze table, silver table, business key)
specs = [
  ("raw_orders",        "clean_orders",        "order_id"),
  ("raw_pos_sales",     "clean_pos_sales",     "sale_id"),
  ("raw_transactions",  "clean_transactions",  "txn_id"),
  ("raw_gl_postings",   "clean_gl_postings",   "gl_id"),
  ("raw_opportunities", "clean_opportunities", "opp_id"),
]

for bronze, silver, key in specs:
    src = (spark.table(f"workspace.bronze.{bronze}")
           .drop("_rescued_data")
           .filter(col(key).isNotNull())
           .dropDuplicates([key]))
    tgt_name = f"workspace.silver.{silver}"
    if not spark.catalog.tableExists(tgt_name):
        src.write.saveAsTable(tgt_name)
    else:
        (DeltaTable.forName(spark, tgt_name).alias("t")
            .merge(src.alias("s"), f"t.{key} = s.{key}")
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    print(silver, spark.table(tgt_name).count())

clean_orders 20000
clean_pos_sales 30000
clean_transactions 20000
clean_gl_postings 10000
clean_opportunities 3000


In [0]:
%sql
DROP TABLE IF EXISTS workspace.silver.clean_orders;

In [0]:
%sql
SHOW TABLES IN workspace.silver;
DESCRIBE workspace.silver.clean_orders;

col_name,data_type,comment
order_id,int,null
cust_id,int,null
order_amt,double,null
order_dt,date,null
last_updated_ts,date,null
_ingest_ts,timestamp,null
_source_file,string,null
_source_system,string,null
# Clustering Information,,
# col_name,data_type,comment
